# Modelling: evidence sufficiency classification
Predict ANSWER (1) versus ESCALATE (0), not the answer itself. Compare three algorithms, four feature sets and two training strategies on identical real validation folds. Protected test data must remain unopened until evaluation.

In [ ]:
from pathlib import Path
import sys, json, time, hashlib, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, confusion_matrix, accuracy_score)
from xgboost import XGBClassifier
SEED=42
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/private').exists()),Path.cwd())
# Set explicitly if your export location differs.
DATA=ROOT/'data/private/final_dataset_v2/development_dataset.csv'
if not DATA.exists():
    candidates=list((ROOT/'data/private').glob('**/development_dataset.csv'))
    candidates=[p for p in candidates if 'backup' not in str(p).lower()]
    if len(candidates)==1: DATA=candidates[0]
    elif (ROOT/'development_dataset.csv').exists(): DATA=ROOT/'development_dataset.csv'
    else: raise FileNotFoundError('Set DATA to the final development CSV; do not use a backup.')
sys.path.insert(0,str(ROOT)); sys.path.insert(0,str(Path.cwd()))
from evidence_features import make_features
print(DATA)

## 1. Verify development data
Administrative columns are for audit only. The test file is never loaded. Metadata supplies expected partition totals, not test observations.

In [ ]:
d=pd.read_csv(DATA,keep_default_na=False,dtype={'record_id':str,'group_id':str})
assert len(d)==1017 and d.record_id.is_unique
assert d.partition.eq('development').all()
assert d.group_id.str.strip().ne('').all()
assert d.target.eq(d.decision_label.map({'ANSWER':1,'ESCALATE':0})).all()
assert d.knowledge_base_version.nunique()==1
assert d.source_type.isin(['Real','Synthetic']).all()
print(pd.crosstab(d.source_type,d.decision_label))
meta = DATA.parent / 'partition_metadata.json'

assert meta.exists(), "Partition metadata not found."

m = json.loads(meta.read_text())

# Validate partition configuration
assert m['seed'] == SEED
assert m['test_answer'] == 25
assert m['test_escalate'] == 175
assert m['quarantined'] == 0
assert m['optimizer_status'] == 0

# Validate development dataset
assert len(d) == 1017
assert d.record_id.is_unique
assert (d.source_type == 'Real').sum() == 717
assert (d.source_type == 'Synthetic').sum() == 300
assert (d.decision_label == 'ANSWER').sum() == 255
assert (d.decision_label == 'ESCALATE').sum() == 762

# Validate knowledge base consistency
assert d.knowledge_base_version.nunique() == 1
assert d.knowledge_base_version.iloc[0] == m['knowledge_base_version']

# Check partition independence without inspecting test features
test_ids = pd.read_csv(
    DATA.parent / 'protected_test_dataset.csv',
    usecols=['record_id', 'group_id'],
    dtype=str
)

assert len(test_ids) == 200
assert test_ids.record_id.is_unique

assert not (
    set(d.record_id.astype(str)) &
    set(test_ids.record_id)
)

assert not (
    set(d.group_id.astype(str)) &
    set(test_ids.group_id)
)

del test_ids

print("Unique development groups:", d.group_id.nunique())
print("Knowledge base version:", m['knowledge_base_version'])
print("All dataset validation checks passed.")

## 2. Feature engineering
Word and character TF-IDF capture vocabulary and spelling; structured intent indicators describe customer needs; method-specific retrieval scores and candidate coverage describe available evidence. Coverage is a proxy, not proof of correct matching. A listed test does not establish real-time availability; populated zero prices are not silently discarded. Missing scores remain missing until fold-local imputation. No source, ID, label or batch is a predictor. Deterministic transformations are in evidence_features.py for reuse at inference.

In [ ]:
X=make_features(d); y=d.target.to_numpy(dtype=int)
S=[c for c in X if c.startswith('s_')]; R=[c for c in X if c.startswith('r_')]
print('Structured:',len(S),'Retrieval:',len(R))
display(X[S+R].describe().T)
assert np.isfinite(X[S+R].fillna(0)).all().all()
FEATURES={'A':(True,[]),'B':(True,R),'C':(False,S+R),'D':(True,S+R)}

## 3. Validation design
Five stratified group folds on real records. Augmentation adds synthetic records only to training, excluding validation groups. Fit TF-IDF, imputation and scaling inside each fold. Group IDs do not guarantee semantic independence. Validation estimates may be optimistic after selecting configurations and thresholds.

In [ ]:
real=np.flatnonzero(d.source_type.eq('Real')); syn=np.flatnonzero(d.source_type.eq('Synthetic'))
cv=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
folds=[]
for tr,va in cv.split(real,y[real],d.group_id.iloc[real]):
    a,b=real[tr],real[va]; groups=set(d.group_id.iloc[b]); extra=np.array([i for i in syn if d.group_id.iloc[i] not in groups],dtype=int)
    assert len(np.unique(y[b]))==2 and len(np.unique(y[a]))==2
    assert not set(d.group_id.iloc[a]) & groups
    folds.append((a,b,np.concatenate([a,extra])))
print(pd.DataFrame([{'fold':i+1,'real_train':len(a),'augmented_train':len(c),'validation':len(b),'validation_answer':sum(y[b])} for i,(a,b,c) in enumerate(folds)]))

## 4. Preprocessing and model factories
Logistic regression and linear SVM use regularized linear decision boundaries. XGBoost uses shallow boosted trees to capture nonlinear interactions. Balanced weights are computed only on the training fold; no resampling of validation. SVM uses margins, not probabilities, and is thresholded at zero by default. XGBoost and logistic regression use positive-class scores, not necessarily calibrated probabilities.

In [ ]:
def build(model,feature,train_y,params=None):
    params=dict(params or {}); weighted=params.pop('weighted',True)
    text,cols=FEATURES[feature]; parts=[]
    if text:
        parts += [('word',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=8000,sublinear_tf=True),'question'),('char',TfidfVectorizer(analyzer='char',ngram_range=(3,5),min_df=2,max_features=12000,sublinear_tf=True),'question')]
    if cols:
        parts.append(('numeric',Pipeline([('fill',SimpleImputer(strategy='constant',fill_value=-1)),('scale',StandardScaler())]),cols))
    w='balanced' if weighted else None
    if model=='LR': est=LogisticRegression(C=1,max_iter=2000,class_weight=w,random_state=SEED,**params) if 'C' not in params else LogisticRegression(max_iter=2000,class_weight=w,random_state=SEED,**params)
    elif model=='SVM': est=LinearSVC(class_weight=w,random_state=SEED,max_iter=10000,**params)
    else:
        opts=dict(n_estimators=150,max_depth=3,learning_rate=.08,subsample=.9,colsample_bytree=.9,n_jobs=2,tree_method='hist',random_state=SEED,eval_metric='logloss',scale_pos_weight=float(sum(train_y==0)/sum(train_y==1)) if weighted else 1.)
        opts.update(params); est=XGBClassifier(**opts)
    return Pipeline([('preprocess',ColumnTransformer(parts,sparse_threshold=1.0)),('classifier',est)])
def scores(pipe, model, z):
    return pipe.decision_function(z) if model=='SVM' else pipe.predict_proba(z)[:,1]
def metrics(t,s,threshold):
    pred=(s>=threshold).astype(int); tn,fp,fn,tp=confusion_matrix(t,pred,labels=[0,1]).ravel()
    return dict(accuracy=accuracy_score(t,pred),macro_f1=f1_score(t,pred,average='macro'),answer_precision=precision_score(t,pred,zero_division=0),answer_recall=recall_score(t,pred,zero_division=0),escalate_recall=tn/(tn+fp),average_precision=average_precision_score(t,s),roc_auc=roc_auc_score(t,s),coverage=pred.mean(),false_answer=int(fp),true_answer=int(tp))
def run(model,feature,strategy,params=None):
    oof=np.full(len(d),np.nan); fold_results=[]
    for a,b,c in folds:
        tr=a if strategy=='real' else c
        pipe=build(model,feature,y[tr],params); pipe.fit(X.iloc[tr],y[tr]); s=scores(pipe,model,X.iloc[b]); oof[b]=s
        fold_results.append(metrics(y[b],s,0 if model=='SVM' else .5))
    assert np.isfinite(oof[real]).all()
    return oof[real],dict(metrics(y[real],oof[real],0 if model=='SVM' else .5),fold_macro_f1_sd=np.std([m['macro_f1'] for m in fold_results]))

## 5. Baseline and comparative experiments
Average precision (AP) summarizes the precision–recall curve using recall-weighted precision; it is not trapezoidal PR-AUC. Compare pooled real out-of-fold predictions. All-escalate baseline has zero automation. A vs B tests added retrieval; B vs D tests added extraction features; C vs D tests added text. Retrieval itself depends on upstream extraction.

In [ ]:
dummy=DummyClassifier(strategy='most_frequent'); dummy.fit(np.zeros((len(real),1)),y[real]); baseline=dummy.predict_proba(np.zeros((len(real),1)))[:,1]
print('All-escalate baseline:',metrics(y[real],baseline,.5))
from itertools import product
results=[]; predictions={}
for model,feature,strategy in product(['LR','SVM','XGB'],FEATURES,['real','augmented']):
    start=time.perf_counter(); key=(model,feature,strategy); p,m=run(*key); predictions[key]=p
    results.append(dict(model=model,feature=feature,strategy=strategy,seconds=time.perf_counter()-start,**m))
    print(key,round(m['average_precision'],3),flush=True)
comparison=pd.DataFrame(results).sort_values('average_precision',ascending=False)
display(comparison.round(4))

In [ ]:
display(comparison.pivot(index=['model','feature'],columns='strategy',values='average_precision').assign(augmentation_delta=lambda t:t.augmented-t.real))
comparison.set_index(['model','feature','strategy']).average_precision.sort_values().plot.barh(figsize=(9,9),title='Real validation average precision'); plt.tight_layout(); plt.show()

## 6. Hyperparameter tuning
For computational efficiency tune the best AP configuration within each algorithm. Small predeclared grids test regularization, class weighting and tree complexity. Retain the untuned configuration as a candidate. This is not nested CV; final independent evaluation is essential.

In [ ]:
grids={'LR':[{}, {'C':.1},{'C':10},{'weighted':False}], 'SVM':[{}, {'C':.1},{'C':10},{'weighted':False}], 'XGB':[{}, {'max_depth':2,'n_estimators':200},{'max_depth':4,'min_child_weight':3},{'weighted':False}]}
tuned=[]; tuned_scores={}
for model in grids:
    best=comparison[comparison.model.eq(model)].iloc[0]; feature=best.feature; strategy=best.strategy
    for params in grids[model]:
        p,m=run(model,feature,strategy,params) if params else (predictions[(model,feature,strategy)],metrics(y[real],predictions[(model,feature,strategy)],0 if model=='SVM' else .5))
        idx=len(tuned); tuned_scores[idx]=p; tuned.append(dict(model=model,feature=feature,strategy=strategy,params=params,**m))
tuning=pd.DataFrame(tuned); display(tuning.sort_values('average_precision',ascending=False))

## 7. Safety-oriented threshold and final candidate selection
Provisional engineering targets (not clinically validated): ANSWER precision ≥ 90%, ESCALATE recall ≥ 95%, at least 10 predicted answers on real OOF cases. Among feasible operating points maximize true ANSWER count, then precision, then AP. If no candidate qualifies, default to all-escalate and report the failure. Thresholds are selected on development predictions; the independent test remains untouched. Confirm these targets with the project supervisor before claiming business acceptance.

In [ ]:
MIN_PRECISION=.90; MIN_ESCALATE_RECALL=.95; MIN_PREDICTED_ANSWER=10
operating=[]
for i,row in tuning.iterrows():
    s=tuned_scores[i]
    for threshold in np.unique(s):
        # Fast screening before calculating complete metrics
        pred=s>=threshold; tp=int(sum(pred & (y[real]==1))); fp=int(sum(pred & (y[real]==0)))
        if tp+fp<MIN_PREDICTED_ANSWER: continue
        if tp/(tp+fp)<MIN_PRECISION or 1-fp/sum(y[real]==0)<MIN_ESCALATE_RECALL: continue
        operating.append(dict(config=i,threshold=float(threshold),**metrics(y[real],s,threshold)))
if operating:
    operating=pd.DataFrame(operating).sort_values(['true_answer','answer_precision','average_precision'],ascending=False)
    chosen=operating.iloc[0]; selected=tuning.loc[int(chosen.config)]; threshold=float(chosen.threshold); fallback=False
    display(operating.head(10)); print('Selected:',selected.to_dict())
else:
    selected=tuning.sort_values('average_precision',ascending=False).iloc[0]; threshold=None; fallback=True
    print('No configuration meets provisional targets. All-escalate policy; diagnostic model only.')
print('Threshold:',threshold,'Fallback:',fallback)

In [ ]:
print("Selected model:", selected.model)
print("Feature set:", selected.feature)
print("Training strategy:", selected.strategy)
print("Hyperparameters:", selected.params)
print("Decision threshold:", repr(float(threshold)))
print("Fallback policy:", fallback)

tr = real if selected.strategy == 'real' else np.arange(len(d))

print("Training records:", len(tr))
print(d.iloc[tr].decision_label.value_counts())

## 8. Retrain and serialize
Retrain selected pipeline on its full allowed development pool. The saved pipeline expects the output of make_features; the threshold and fallback policy are separate. Artifacts are private because TF-IDF vocabulary may contain sensitive tokens. No test evaluation is conducted here. Inference must also enforce evidence validity and safety guards before invoking the response LLM.

In [ ]:
tr=real if selected.strategy=='real' else np.arange(len(d))
final_model=build(selected.model,selected.feature,y[tr],selected.params); final_model.fit(X.iloc[tr],y[tr])
policy=dict(model=str(selected.model),feature=str(selected.feature),strategy=str(selected.strategy),params=selected.params,threshold=threshold,all_escalate=fallback,score_type='margin' if selected.model=='SVM' else 'positive_class_score',seed=SEED,kb_version=str(d.knowledge_base_version.iloc[0]),development_sha256=hashlib.sha256(DATA.read_bytes()).hexdigest(),versions={k:importlib.metadata.version(k) for k in ['scikit-learn','xgboost','pandas','numpy','joblib']})
OUT=ROOT/'data/private/modelling_v1'; OUT.mkdir(parents=True,exist_ok=True)
joblib.dump(final_model,OUT/'classifier.joblib')
(OUT/'policy.json').write_text(json.dumps(policy,indent=2))
comparison.to_csv(OUT/'comparison.csv',index=False); tuning.to_csv(OUT/'tuning.csv',index=False)
if not fallback: operating.to_csv(OUT/'operating_points.csv',index=False)
print(policy); print('Saved:',OUT)
# Verify serialized pipeline reproduces scores
reloaded=joblib.load(OUT/'classifier.joblib'); assert np.allclose(scores(reloaded,selected.model,X.iloc[:10]),scores(final_model,selected.model,X.iloc[:10]))

In [ ]:
for p in sorted(OUT.iterdir()):
    print(f"{p.name}: {p.stat().st_size / 1024:.2f} KB")

## 9. Independent Test Evaluation

The final trained model and frozen decision policy
are evaluated on the protected real-only test dataset.
No further model fitting, hyperparameter tuning or
threshold optimization is performed.

In [ ]:
TEST = DATA.parent / 'protected_test_dataset.csv'

assert TEST.exists(), "Protected test dataset not found."

test = pd.read_csv(
    TEST,
    keep_default_na=False,
    dtype={'record_id': str, 'group_id': str}
)

# Load frozen model and decision policy
frozen_model = joblib.load(OUT / 'classifier.joblib')

frozen_policy = json.loads(
    (OUT / 'policy.json').read_text()
)

# Validate test dataset
assert len(test) == 200
assert test.record_id.is_unique
assert test.source_type.eq('Real').all()
assert test.decision_label.eq('ANSWER').sum() == 25
assert test.decision_label.eq('ESCALATE').sum() == 175

# Confirm partition independence
assert not (
    set(test.record_id) &
    set(d.record_id.astype(str))
)

assert not (
    set(test.group_id) &
    set(d.group_id.astype(str))
)

# Confirm knowledge base consistency
assert test.knowledge_base_version.eq(
    frozen_policy['kb_version']
).all()

# Confirm development dataset has not changed
assert hashlib.sha256(
    DATA.read_bytes()
).hexdigest() == frozen_policy['development_sha256']

print("Protected test records:", len(test))
print(test.decision_label.value_counts())
print("\nFrozen decision policy:")
print(json.dumps(frozen_policy, indent=2))

print("\nAll independent test validation checks passed.")

In [ ]:
# Generate features using the same feature engineering function
X_test = make_features(test)

# Encode ground truth
y_test = test['decision_label'].eq('ANSWER').astype(int).to_numpy()

# Generate scores using the frozen classifier
test_scores = scores(
    frozen_model,
    frozen_policy['model'],
    X_test
)

# Apply the frozen decision threshold
test_threshold = frozen_policy['threshold']

if frozen_policy['all_escalate']:
    y_pred = np.zeros(len(test), dtype=int)
else:
    y_pred = (test_scores >= test_threshold).astype(int)

print("Test feature dimensions:", X_test.shape)
print("Decision threshold:", test_threshold)
print("Total predictions:", len(y_pred))
print("Predicted ANSWER:", int(y_pred.sum()))
print("Predicted ESCALATE:", int((y_pred == 0).sum()))

In [ ]:
tn, fp, fn, tp = confusion_matrix(
    y_test, y_pred, labels=[0, 1]
).ravel()

test_metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'macro_f1': f1_score(
        y_test, y_pred, average='macro'
    ),
    'answer_precision': precision_score(
        y_test, y_pred, zero_division=0
    ),
    'answer_recall': recall_score(
        y_test, y_pred, zero_division=0
    ),
    'escalate_recall': recall_score(
        y_test, y_pred, pos_label=0, zero_division=0
    ),
    'average_precision': average_precision_score(
        y_test, test_scores
    ),
    'roc_auc': roc_auc_score(
        y_test, test_scores
    ),
    'coverage': float(y_pred.mean()),
    'false_answer': int(fp),
    'true_answer': int(tp),
    'false_escalate': int(fn),
    'true_escalate': int(tn)
}

display(
    pd.DataFrame.from_dict(
        test_metrics,
        orient='index',
        columns=['Independent test']
    ).round(6)
)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=[1, 0],
    display_labels=['ANSWER', 'ESCALATE'],
    cmap='Blues',
    colorbar=False,
    ax=ax
)

ax.set_title('Independent Test Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
development_metrics = {
    'accuracy': 678 / 717,
    'macro_f1': 0.725651,
    'answer_precision': 18 / 20,
    'answer_recall': 18 / 55,
    'escalate_recall': 660 / 662,
    'average_precision': 0.642868,
    'roc_auc': 0.952019,
    'coverage': 20 / 717,
    'false_answer': 2,
    'true_answer': 18,
    'false_escalate': 37,
    'true_escalate': 660
}

evaluation_comparison = pd.DataFrame({
    'Development OOF': development_metrics,
    'Independent test': test_metrics
})

display(evaluation_comparison.round(6))

In [ ]:
acceptance = pd.DataFrame({
    'Criterion': [
        'ANSWER precision',
        'ESCALATE recall',
        'Predicted ANSWER count'
    ],
    'Required': [0.90, 0.95, 10],
    'Observed': [
        test_metrics['answer_precision'],
        test_metrics['escalate_recall'],
        int(y_pred.sum())
    ]
})

acceptance['Met'] = (
    acceptance['Observed'] >= acceptance['Required']
)

display(acceptance)

print(
    "All provisional criteria met:",
    bool(acceptance['Met'].all())
)

## 10. Error Analysis

In [ ]:
audit = test.copy()

audit['model_score'] = test_scores

audit['predicted_decision'] = np.where(
    y_pred == 1, 'ANSWER', 'ESCALATE'
)

audit['outcome'] = np.select(
    [
        (y_test == 1) & (y_pred == 1),
        (y_test == 0) & (y_pred == 1),
        (y_test == 1) & (y_pred == 0)
    ],
    [
        'TRUE_ANSWER',
        'FALSE_ANSWER',
        'FALSE_ESCALATE'
    ],
    default='TRUE_ESCALATE'
)

audit['threshold_distance'] = (
    audit['model_score'] - test_threshold
)

cols = [
    'record_id',
    'question',
    'intent_group',
    'requested_fields',
    'retrieval_status',
    'retrieved_test_name',
    'decision_label',
    'predicted_decision',
    'model_score',
    'threshold_distance',
    'outcome'
]

print("FALSE ANSWER CASES")

display(
    audit.loc[
        audit.outcome.eq('FALSE_ANSWER'), cols
    ]
)

print("FALSE ESCALATE CASES")

display(
    audit.loc[
        audit.outcome.eq('FALSE_ESCALATE'), cols
    ].sort_values('model_score', ascending=False)
)

print("TRUE ANSWER CASES")

display(
    audit.loc[
        audit.outcome.eq('TRUE_ANSWER'), cols
    ].sort_values('model_score', ascending=False)
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bins = np.linspace(0, 1, 26)

ax.hist(
    test_scores[y_test == 0],
    bins=bins,
    alpha=0.5,
    density=True,
    label='Actual ESCALATE'
)

ax.hist(
    test_scores[y_test == 1],
    bins=bins,
    alpha=0.5,
    density=True,
    label='Actual ANSWER'
)

ax.axvline(
    test_threshold,
    color='black',
    linestyle='--',
    label=f'Frozen threshold: {test_threshold:.4f}'
)

ax.set_xlabel('Model score')
ax.set_ylabel('Density')
ax.set_title('Independent Test Score Distributions')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
EVAL = OUT / 'evaluation'
EVAL.mkdir(parents=True, exist_ok=True)

evaluation_comparison.to_csv(
    EVAL / 'evaluation_comparison.csv'
)

acceptance.to_csv(
    EVAL / 'acceptance_criteria.csv',
    index=False
)

audit[cols].to_csv(
    EVAL / 'test_predictions.csv',
    index=False
)

audit.loc[
    audit.outcome.isin(
        ['FALSE_ANSWER', 'FALSE_ESCALATE']
    ),
    cols
].to_csv(
    EVAL / 'misclassified_cases.csv',
    index=False
)

(EVAL / 'test_metrics.json').write_text(
    json.dumps(test_metrics, indent=2)
)

print("Evaluation artifacts saved:", EVAL)

In [ ]:
false_answer = audit.loc[
    audit.outcome.eq('FALSE_ANSWER')
].iloc[0]

print("=" * 80)
print("ORIGINAL QUESTION")
print("=" * 80)
print(false_answer['question'])

print("\n" + "=" * 80)
print("LLM EXTRACTION")
print("=" * 80)
print(json.dumps(
    json.loads(false_answer['extraction_json']),
    indent=2,
    ensure_ascii=False
))

print("\n" + "=" * 80)
print("RETRIEVED EVIDENCE")
print("=" * 80)
print(json.dumps(
    json.loads(false_answer['candidate_evidence_json']),
    indent=2,
    ensure_ascii=False
))

print("\n" + "=" * 80)
print("ANNOTATION")
print("=" * 80)
print("Actual decision:", false_answer['decision_label'])
print("Label reason:", false_answer['label_reason'])
print("Predicted decision:", false_answer['predicted_decision'])
print("Model score:", false_answer['model_score'])

In [ ]:
extraction = json.loads(false_answer['extraction_json'])
evidence = json.loads(false_answer['candidate_evidence_json'])

print("EXTRACTION SUMMARY")
print("Intents:", extraction['intent_group'])
print("Ambiguity:", extraction['ambiguity_flag'])
print("Context dependent:", extraction['context_dependent'])

print("\nREQUESTED TESTS AND RETRIEVED CANDIDATES")

for req in extraction['requested_tests']:
    print("\nRequested test:", req['test_name'])
    print("Requested fields:", req['requested_fields'])

    candidates = [
        c for c in evidence
        if c['search_query'] == req['test_name']
    ]

    if not candidates:
        print("NO CANDIDATES FOUND")

    for c in candidates:
        print(
            f"Rank {c['rank']} | "
            f"Code: {c['test_code']} | "
            f"Test: {c['test_name']} | "
            f"Price: {c['price']} | "
            f"Method: {c['retrieval_method']} | "
            f"Score: {c['retrieval_score']}"
        )

### Error Analysis and Implications for System Design

The independent test evaluation identified one false ANSWER prediction involving a customer enquiry requesting prices for three laboratory investigations. Although the extraction component correctly identified the PRICE intent and three requested investigations, it failed to identify ambiguity in the request for 24-hour urine albumin protein levels.

The hybrid retrieval component successfully retrieved candidates for all requested investigations. However, the highest-ranked candidate for the ambiguous urine investigation was serum albumin, representing a specimen mismatch. Other retrieved candidates represented different investigations, while a potentially appropriate candidate was ranked third.

Despite these limitations, the classifier assigned an ANSWER score of 0.9582, exceeding the predetermined decision threshold of 0.7423.

This finding demonstrates that successful retrieval and the availability of requested information do not necessarily establish evidence sufficiency. It also highlights the potential for errors to propagate across interconnected components of a retrieval-augmented generation system.

The findings motivate the incorporation of additional evidence verification and fail-safe escalation mechanisms before response generation. The original classifier and decision threshold remain unchanged to preserve the integrity of the independent evaluation. Any subsequent improvements must be assessed using additional independent evaluation data.